# A and C Optuna

In [1]:
# %% [markdown]
# # 02 — Dense KAN (3-Seed Reproducibility)
#
# Fully-connected KAN baseline using efficient-kan's KANLinear directly
# (not the efficient_kan.KAN wrapper -- that wrapper has no hook for
# inserting layers between KANLinear blocks, and this design now requires
# BatchNorm between every layer; see below).
#
# Layer widths are DERIVED PER DATASET from the taxonomy (same as Sparse
# KAN and Dense MLP): [n_features, n_subthemes, n_themes, 1]. agg_means
# and agg_full_moments have genuinely different subtheme counts (128 vs
# 331), so a single shared width pair would be wrong for one of them.
#
# ═══════════════════════════════════════════════════════════════════════
# MAJOR REVISION — grid adaptation removed, BatchNorm added
# ═══════════════════════════════════════════════════════════════════════
#
# An earlier version of this notebook called efficient-kan's update_grid()
# at epochs [1, 5, 20] to reposition each layer's B-spline knots to fit
# the data. This was investigated in depth and abandoned:
#
#   1. update_grid()'s internal curve2coeff() least-squares refit was
#      found, via direct diagnostic, to produce NaN/inf on legitimate
#      active edges -- confirmed independent of grid_eps.
#
#   2. A corruption audit across 48 trained checkpoints from that version
#      found 25/48 (52%) showed NaN propagation during training.
#
#   3. A follow-up per-node activation-range measurement found the ROOT
#      CAUSE: hidden-layer activation scale varies by orders of magnitude
#      (typical spread 1-5, worst observed case >400) depending on
#      dataset, target, and training progress. No fixed OR adaptively-
#      repositioned grid can be correct for both regimes at once.
#
# THE FIX: nn.BatchNorm1d is inserted between every KAN layer. BatchNorm
# forces each unit's activation toward zero-mean/unit-variance BEFORE the
# affine step, regardless of feature count or target scale -- the
# invariant a fixed grid range was always silently assuming. With that
# invariant genuinely enforced, grid_range=[-5.5, 5.5] is correct for
# every layer, permanently, and update_grid() is no longer needed at all.
#
# BatchNorm (not LayerNorm) was chosen deliberately: LayerNorm's
# statistics are computed ACROSS UNITS within one sample, entangling
# every hidden unit's contribution with every other's -- fatal for this
# architecture's interpretability goals (Sparse KAN's whole point is
# isolable per-subtheme/theme contributions). BatchNorm's statistics are
# computed ACROSS THE BATCH, per unit, so unit i's output depends only on
# unit i's own values. At inference it reduces to a fixed per-unit affine
# map, in principle foldable into the adjacent spline for clean symbolic
# extraction.
#
# grid_eps has been removed entirely (it was only ever read inside
# update_grid()). GRID_RANGE is now fixed for the model's entire lifetime,
# same as it always was for layer 0.
#
# ACTIVATION MONITORING: training.py gained one small, purely additive
# parameter, epoch_callback (called at the same cadence as the existing
# loss printout, zero effect if unset -- see training.py docstring). This
# notebook uses it to print each layer's pre- and post-BatchNorm max|x|
# and std on a fixed probe batch, live, during the final retrain of each
# of the 48 configs -- so a bad config is visible within minutes, not
# after a full overnight run finishes.
#
# Fixed architecture (not tuned):
#   - grid_size = 14, spline_order = 3 (cubic B-splines)
#   - grid_range = [-5.5, 5.5], fixed at construction, never touched again
#   - BatchNorm1d(n_subthemes) between layer0 and layer1
#   - BatchNorm1d(n_themes) between layer1 and layer2
#   - gradient clipping = 1.0
#
# 3 seeds × 4 splits × 2 datasets × 2 targets = 48 runs
# Each seed runs the FULL pipeline independently:
#   - Optuna (40 trials) with seeded TPE sampler
#   - Final retrain with best params
#   - Evaluation on train/val/test
#
# Regularisation: edge-pruning L1 (spline_scaler + base_weight, KAN layers
# only -- BatchNorm's gamma/beta are never penalised)
#
# Target notes:
#   binary:     y_binary (minret_5d_pct < -2.0), BCEWithLogitsLoss,
#               early stop on AUC (uniform across every binary-target
#               model in this project -- see training.py docstring).
#   continuous: minret_5d_pct (raw percentage, NOT a z-score),
#               HuberLoss(delta=<per-split value from huber_delta.json>),
#               early stop on R².
#   Both use Optuna direction="maximize".
#
# COLLAPSE HANDLING: detect-and-flag only, no automatic retry. Not
# checked inline -- deferred to the standalone Collapse_Audit notebook,
# run once after all models finish.
#
# Results save to Drive per-seed as they go -- safe against disconnection.
#
# Estimated runtime on T4: ~3 seeds × estimated per-seed time (see timing
# cell). No grid-update overhead now, so likely somewhat faster than the
# earlier version.
# %%
# ── COLAB SETUP ──
!pip install -q git+https://github.com/Blealtan/efficient-kan.git optuna

from google.colab import drive
drive.mount("/content/drive")

import sys
sys.path.insert(0, "/content/drive/MyDrive/Thesis/Code")

# %%
import json
import numpy as np
import pandas as pd
import random
import time
import torch
import torch.nn as nn
import optuna
from pathlib import Path
from efficient_kan import KANLinear

from data_utils import load_split, get_dataloaders, get_device, load_theme_assignment
from training import train_model, save_checkpoint
from evaluation import (
    evaluate_model, save_predictions, compute_calibration,
    load_predictions, run_full_backtest,
)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

SPLITS_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/04_splits")
THEMES_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/05_themes")
HUBER_DELTA_PATH = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/03_targets/huber_delta.json")

RESULTS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/dense_kan")

DATASETS     = ["agg_full_moments", "agg_means"]
TARGET_TYPES = ["binary", "continuous"]
ALL_SPLITS   = ["Split_A", "Split_B", "Split_C", "Split_D"]

SEEDS = [42, 123, 456]

# ── Fixed architecture (not tuned) ──
GRID_SIZE    = 14
SPLINE_ORDER = 3
GRID_RANGE   = [-5.5, 5.5]     # fixed for the model's ENTIRE lifetime,
                               # every layer, no adaptation, ever.

ACTIVATION_PROBE_N = 2048      # rows drawn once per run for the live
                               # activation-monitoring printout below

N_TRIALS = 40

with open(HUBER_DELTA_PATH) as f:
    HUBER_DELTAS = json.load(f)["deltas"]

def get_huber_delta(split_name):
    return HUBER_DELTAS[f"{split_name}/market"]


# ═══════════════════════════════════════════════════════════════════════════════
# REPRODUCIBILITY
# ═══════════════════════════════════════════════════════════════════════════════

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ═══════════════════════════════════════════════════════════════════════════════
# ARCHITECTURE — widths derived per dataset from the taxonomy
# ═══════════════════════════════════════════════════════════════════════════════

def get_architecture_widths(dataset):
    tax = load_theme_assignment(dataset, THEMES_DIR)
    n_subthemes = tax["subtheme_id"].nunique()
    n_themes    = tax["theme_id"].nunique()
    return n_subthemes, n_themes

ARCHITECTURE = {ds: get_architecture_widths(ds) for ds in DATASETS}

print("=" * 70)
print("  ARCHITECTURE WIDTHS (derived from taxonomy, per dataset)")
print("=" * 70)
for ds, (n_sub, n_th) in ARCHITECTURE.items():
    print(f"  {ds:<20} n_subthemes={n_sub:>4}   n_themes={n_th:>3}")


# ═══════════════════════════════════════════════════════════════════════════════
# MODEL — three KANLinear layers with BatchNorm in between
# ═══════════════════════════════════════════════════════════════════════════════
# Built directly from KANLinear rather than efficient_kan.KAN's wrapper
# class, since that wrapper has no mechanism for inserting layers between
# its KANLinear blocks. Attribute names (layer0/layer1/layer2) deliberately
# match SparseKAN's naming for consistency between the two architectures.

class DenseKANBN(nn.Module):
    def __init__(self, n_features, n_subthemes, n_themes,
                 grid_size, spline_order, grid_range):
        super().__init__()
        self.n_features  = n_features
        self.n_subthemes = n_subthemes
        self.n_themes    = n_themes
        self.grid_size    = grid_size
        self.spline_order = spline_order
        self.grid_range   = grid_range

        self.layer0 = KANLinear(
            n_features, n_subthemes,
            grid_size=grid_size, spline_order=spline_order,
            scale_noise=0.1, scale_base=1.0, scale_spline=1.0,
            grid_range=grid_range,
        )
        self.bn1 = nn.BatchNorm1d(n_subthemes, affine=False, eps=0.1)

        self.layer1 = KANLinear(
            n_subthemes, n_themes,
            grid_size=grid_size, spline_order=spline_order,
            scale_noise=0.1, scale_base=1.0, scale_spline=1.0,
            grid_range=grid_range,
        )
        self.bn2 = nn.BatchNorm1d(n_themes, affine=False, eps=0.1)

        self.layer2 = KANLinear(
            n_themes, 1,
            grid_size=grid_size, spline_order=spline_order,
            scale_noise=0.1, scale_base=1.0, scale_spline=1.0,
            grid_range=grid_range,
        )

    def forward(self, x):
        x = self.layer0(x)
        x = self.bn1(x)
        x = torch.clamp(x, min=-5.0, max=5.0)  # Absolute grid boundary guarantee
        
        x = self.layer1(x)
        x = self.bn2(x)
        x = torch.clamp(x, min=-5.0, max=5.0)  # Absolute grid boundary guarantee
        
        x = self.layer2(x)
        return x

    def all_kan_layers(self):
        """Yields only the three KANLinear layers, NOT the BatchNorm
        layers -- used by edge_pruning_l1 so BatchNorm's gamma/beta are
        never penalised."""
        yield self.layer0
        yield self.layer1
        yield self.layer2


def make_dense_kan(n_features, n_subthemes, n_themes):
    return DenseKANBN(
        n_features, n_subthemes, n_themes,
        grid_size=GRID_SIZE, spline_order=SPLINE_ORDER, grid_range=GRID_RANGE,
    )


# ═══════════════════════════════════════════════════════════════════════════════
# ACTIVATION MONITORING — live printout during the final retrain only
# ═══════════════════════════════════════════════════════════════════════════════
# Uses training.py's new epoch_callback hook (see that file's docstring).
# Fires only when verbose=True, which is the case for the final retrain
# but NOT the (verbose=False) Optuna search -- so this gives one printout
# per final model, at the log_every cadence, without flooding the log
# during the 40-trial search phase.

def make_activation_callback(probe_batch, device):
    @torch.no_grad()
    def callback(model, epoch):
        model.eval()
        x0 = probe_batch.to(device)

        x1 = model.layer0(x0)                        # <-- restored
        x1n_preclamp = model.bn1(x1)
        x1n = torch.clamp(x1n_preclamp, min=-5.0, max=5.0)

        x2 = model.layer1(x1n)
        x2n_preclamp = model.bn2(x2)
        x2n = torch.clamp(x2n_preclamp, min=-5.0, max=5.0)

        model.train()

        print(f"    [activations @ epoch {epoch}]  "
              f"L1 pre-BN max|x|={x1.abs().max().item():7.2f} std={x1.std().item():6.3f}  "
              f"post-BN max|x|={x1n.abs().max().item():5.2f} std={x1n.std().item():5.3f}  |  "
              f"L2 pre-BN max|x|={x2.abs().max().item():7.2f} std={x2.std().item():6.3f}  "
              f"post-BN max|x|={x2n.abs().max().item():5.2f} std={x2n.std().item():5.3f}")

        if model.bn1.weight is not None:
            print(f"    [BN params]  "
                  f"bn1 gamma: max={model.bn1.weight.abs().max().item():.3f} "
                  f"mean={model.bn1.weight.abs().mean().item():.3f}  |  "
                  f"bn2 gamma: max={model.bn2.weight.abs().max().item():.3f} "
                  f"mean={model.bn2.weight.abs().mean().item():.3f}")
    return callback


# ═══════════════════════════════════════════════════════════════════════════════
# EDGE-PRUNING L1 REGULARISATION
# ═══════════════════════════════════════════════════════════════════════════════

def edge_pruning_l1(model):
    """L1 on spline_scaler and base_weight, KAN layers only (BatchNorm's
    gamma/beta are never penalised -- see all_kan_layers())."""
    loss = torch.tensor(0.0, device=next(model.parameters()).device)
    for layer in model.all_kan_layers():
        if hasattr(layer, 'spline_scaler'):
            loss = loss + layer.spline_scaler.abs().sum()
        else:
            loss = loss + layer.spline_weight.abs().mean(dim=-1).sum()
        loss = loss + layer.base_weight.abs().sum()
    return loss


# ═══════════════════════════════════════════════════════════════════════════════
# MODEL FACTORY FOR OPTUNA
# ═══════════════════════════════════════════════════════════════════════════════

def make_model_factory(n_features, n_subthemes, n_themes, huber_delta, target_type):
    """
    4 hyperparameters, same search space as before:
        lr:           [1e-4, 1e-2]    log
        weight_decay: [1e-6, 1e-2]    log
        batch_size:   [64, 128, 256]  categorical
        reg_weight:   [1e-6, 1e-2]    log
    """
    def factory(trial):
        lr           = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
        batch_size   = trial.suggest_categorical("batch_size", [64, 128, 256])
        reg_weight   = trial.suggest_float("reg_weight", 1e-6, 1e-2, log=True)

        model = make_dense_kan(n_features, n_subthemes, n_themes)

        train_kwargs = {
            "lr":           lr,
            "weight_decay": weight_decay,
            "reg_fn":       edge_pruning_l1,
            "reg_weight":   reg_weight,
            "n_epochs":     300,
            "patience":     20,
        }
        if target_type == "continuous":
            train_kwargs["huber_delta"] = huber_delta

        return model, train_kwargs
    return factory


# ═══════════════════════════════════════════════════════════════════════════════
# SINGLE EXPERIMENT
# ═══════════════════════════════════════════════════════════════════════════════

def run_single_experiment(split_name, dataset, target_type, device,
                          seed, seed_results_dir):
    model_name = f"dense_kan_{dataset}"

    print(f"\n{'─'*60}")
    print(f"  seed={seed} / {dataset} / {split_name} / {target_type}")
    print(f"{'─'*60}")

    data       = load_split(split_name, dataset, SPLITS_DIR)
    n_features = data["n_features"]
    n_subthemes, n_themes = ARCHITECTURE[dataset]

    n_pos      = data["y_train"].sum()
    n_neg      = len(data["y_train"]) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32)

    huber_delta = get_huber_delta(split_name) if target_type == "continuous" else None
    metric_name = "AUC" if target_type == "binary" else "R²"

    # ── Fixed probe batch for live activation monitoring during the
    # final retrain (see make_activation_callback). Not used for anything
    # that affects training -- pure diagnostic, drawn once per experiment. ──
    rng = np.random.default_rng(seed)
    n_avail = data["X_train"].shape[0]
    probe_idx = rng.choice(n_avail, size=min(ACTIVATION_PROBE_N, n_avail), replace=False)
    activation_probe = torch.tensor(data["X_train"][probe_idx], dtype=torch.float32)

    # ── OPTUNA SEARCH (seeded TPE sampler) ──
    factory = make_model_factory(n_features, n_subthemes, n_themes, huber_delta, target_type)

    study_path = (seed_results_dir / "optuna"
                  / f"{model_name}_{target_type}_{split_name}.db")
    study_path.parent.mkdir(parents=True, exist_ok=True)

    study = optuna.create_study(
        study_name=f"{model_name}_{target_type}_{split_name}_seed{seed}",
        storage=f"sqlite:///{study_path}",
        direction="maximize",
        load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=seed),
    )

    def objective(trial):
        model, train_kwargs = factory(trial)
        batch_size = trial.params["batch_size"]

        loaders = get_dataloaders(
            split_name, dataset, SPLITS_DIR,
            target_type=target_type,
            batch_size=batch_size,
        )

        result = train_model(
            model=model,
            train_loader=loaders["train"],
            val_loader=loaders["val"],
            device=device,
            target_type=target_type,
            pos_weight=pos_weight if target_type == "binary" else None,
            verbose=False,        # epoch_callback never fires during search
            **train_kwargs,
        )

        return result["best_val_metric"]

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    existing  = sum(1 for t in study.trials
                    if t.state == optuna.trial.TrialState.COMPLETE)
    remaining = max(0, N_TRIALS - existing)

    if remaining > 0:
        print(f"  Running {remaining} Optuna trials ({existing} already complete)")
        study.optimize(objective, n_trials=remaining, show_progress_bar=True)
    else:
        print(f"  Study already has {existing} completed trials — skipping Optuna")

    best_params = study.best_params
    print(f"  Optuna best {metric_name}: {study.best_value:.4f}")
    print(f"  Best params: {best_params}")

    # ── FINAL TRAINING with best params ──
    batch_size = best_params.get("batch_size", 128)
    loaders    = get_dataloaders(
        split_name, dataset, SPLITS_DIR,
        target_type=target_type,
        batch_size=batch_size,
    )

    model = make_dense_kan(n_features, n_subthemes, n_themes)
    activation_callback = make_activation_callback(activation_probe, device)

    final_train_kwargs = dict(
        lr=best_params["lr"],
        weight_decay=best_params["weight_decay"],
        reg_fn=edge_pruning_l1,
        reg_weight=best_params["reg_weight"],
        pos_weight=pos_weight if target_type == "binary" else None,
        n_epochs=300,
        patience=20,
        verbose=True,             # activation_callback WILL fire, per log_every
        log_every=20,
        epoch_callback=activation_callback,
    )
    if target_type == "continuous":
        final_train_kwargs["huber_delta"] = huber_delta

    result = train_model(
        model=model,
        train_loader=loaders["train"],
        val_loader=loaders["val"],
        device=device,
        target_type=target_type,
        **final_train_kwargs,
    )

    # ── EVALUATE on train/val/test ──
    all_metrics = {}
    for part in ["train", "val", "test"]:
        metrics = evaluate_model(
            model, loaders[part], device, target_type,
            y_true_binary=data[f"y_{part}"] if target_type == "continuous" else None,
        )

        if target_type == "binary":
            cal = compute_calibration(metrics["y_true"], metrics["y_prob"])
            metrics["ece"] = cal["ece"]

        save_predictions(
            model_name=model_name,
            split_name=split_name,
            target_type=target_type,
            part=part,
            dates=data[f"dates_{part}"],
            returns=data[f"returns_{part}"],
            metrics=metrics,
            hyperparameters={**best_params, "seed": seed, "huber_delta": huber_delta}
                            if part == "test" else None,
            results_dir=seed_results_dir,
            y_true_binary=data[f"y_{part}"] if target_type == "continuous" else None,
        )

        all_metrics[part] = metrics

    # ── SAVE CHECKPOINT ──
    ckpt_dir = seed_results_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    save_checkpoint(
        model=model,
        train_result=result,
        hyperparameters={**best_params, "seed": seed, "huber_delta": huber_delta},
        model_config={
            "type":          "DenseKANBN",
            "dataset":       dataset,
            "target_type":   target_type,
            "n_features":    n_features,
            "n_subthemes":   n_subthemes,
            "n_themes":      n_themes,
            "grid_size":     GRID_SIZE,
            "spline_order":  SPLINE_ORDER,
            "grid_range":    GRID_RANGE,
        },
        path=ckpt_dir / f"{model_name}_{target_type}_{split_name}.pt",
    )

    # ── PRINT SUMMARY ──
    if target_type == "binary":
        print(f"\n  Results (seed={seed}):")
        print(f"    Train AUC: {all_metrics['train']['auc']:.4f}")
        print(f"    Val AUC:   {all_metrics['val']['auc']:.4f}")
        print(f"    Test AUC:  {all_metrics['test']['auc']:.4f}")
        print(f"    Gap:       "
              f"{all_metrics['train']['auc'] - all_metrics['test']['auc']:+.4f}")
    else:
        print(f"\n  Results (seed={seed}, huber_delta={huber_delta:.4f}):")
        print(f"    Train R²:    {all_metrics['train']['r2']:.4f}  "
              f"(MSE={all_metrics['train']['mse']:.4f})")
        print(f"    Val R²:      {all_metrics['val']['r2']:.4f}  "
              f"(MSE={all_metrics['val']['mse']:.4f})")
        print(f"    Test R²:     {all_metrics['test']['r2']:.4f}  "
              f"(MSE={all_metrics['test']['mse']:.4f})")
        print(f"    Derived AUC: {all_metrics['test']['derived_auc']:.4f}")
        print(f"    Pred std:    {all_metrics['test']['pred_std']:.4f}  "
              f"(collapse check deferred to Collapse_Audit.ipynb)")

    return {
        "best_params": best_params,
        "metrics":     all_metrics,
        "best_epoch":  result["best_epoch"],
        "total_time":  result["total_time"],
    }


# %% [markdown]
# ## Run All Experiments (3 Seeds × 16 Configurations)

# %%
device     = get_device()
configs    = len(DATASETS) * len(TARGET_TYPES) * len(ALL_SPLITS)
total_runs = len(SEEDS) * configs

n_params_by_dataset = {}
for ds in DATASETS:
    n_feat = load_split(ALL_SPLITS[0], ds, SPLITS_DIR)["n_features"]
    n_sub, n_th = ARCHITECTURE[ds]
    n_params_by_dataset[ds] = sum(
        p.numel() for p in make_dense_kan(n_feat, n_sub, n_th).parameters()
    )

print("=" * 70)
print(f"  DENSE KAN: 3 Seeds × 4 Splits × 2 Datasets × 2 Targets = {total_runs} runs")
print(f"  Seeds: {SEEDS}")
for ds in DATASETS:
    n_sub, n_th = ARCHITECTURE[ds]
    print(f"  {ds:<20} widths=[n_feat, {n_sub}, {n_th}, 1]   "
          f"params={n_params_by_dataset[ds]:,}  (incl. BatchNorm gamma/beta)")
print(f"  Grid: G={GRID_SIZE}, K={SPLINE_ORDER}, range={GRID_RANGE} "
      f"(FIXED, no grid adaptation)")
print(f"  BatchNorm1d between every KAN layer")
print(f"  Loss: binary=BCEWithLogitsLoss, "
      f"continuous=HuberLoss(delta=per-split, see huber_delta.json)")
print(f"  Early stop: binary=AUC, continuous=R²  (both maximize)")
print(f"  Optuna: {N_TRIALS} trials per run, seeded TPE sampler")
print(f"  Collapse handling: detect-and-flag post-hoc (Collapse_Audit.ipynb), no retry")
print(f"  Results saving to: {RESULTS_DIR}/seed_*/")
print("=" * 70)

all_results       = []
best_params_store = {}
completed         = 0
failed            = 0
total_start       = time.time()

for seed in SEEDS:
    set_seed(seed)
    seed_results_dir = RESULTS_DIR / f"seed_{seed}"

    print(f"\n\n{'═'*70}")
    print(f"  SEED {seed} — saving to {seed_results_dir}")
    print(f"{'═'*70}")

    for dataset in DATASETS:
        for target_type in TARGET_TYPES:
            for split_name in ALL_SPLITS:
                try:
                    exp = run_single_experiment(
                        split_name, dataset, target_type, device,
                        seed=seed,
                        seed_results_dir=seed_results_dir,
                    )

                    all_results.append({
                        "seed":    seed,
                        "dataset": dataset,
                        "split":   split_name,
                        "target":  target_type,
                        **{f"test_{k}": v for k, v in exp["metrics"]["test"].items()
                           if not isinstance(v, np.ndarray)},
                        "best_epoch": exp["best_epoch"],
                        "time_s":     exp["total_time"],
                    })

                    key = (seed, dataset, target_type, split_name)
                    best_params_store[key] = exp["best_params"]
                    completed += 1

                    elapsed = time.time() - total_start
                    rate = elapsed / completed
                    remaining_est = rate * (total_runs - completed)
                    print(f"\n  ✓ Completed {completed}/{total_runs}  "
                          f"({elapsed/60:.0f}min elapsed, "
                          f"~{remaining_est/60:.0f}min remaining)")

                except Exception as e:
                    failed += 1
                    print(f"\n  ✗ FAILED ({failed}): seed={seed} "
                          f"{dataset}/{split_name}/{target_type}: {e}")
                    import traceback
                    traceback.print_exc()
                    continue

total_time = time.time() - total_start
print(f"\n\n{'='*70}")
print(f"  FINISHED: {completed}/{total_runs} completed, {failed} failed")
print(f"  Total time: {total_time/60:.1f} minutes ({total_time/3600:.1f} hours)")
print(f"{'='*70}")


# %% [markdown]
# ## Cross-Seed Summary
#
# The collapse audit (Collapse_Audit.ipynb) must be run AFTER this notebook
# finishes. Once it exists, re-run this cell to exclude flagged runs.

# %%
if all_results:
    results_df = pd.DataFrame(all_results)

    audit_path = RESULTS_DIR.parent / "collapse_audit.csv"
    if audit_path.exists():
        audit_df = pd.read_csv(audit_path)
        sub_audit = audit_df[audit_df.model == "dense_kan"][
            ["dataset", "target_type", "split", "seed", "collapsed"]
        ].rename(columns={"target_type": "target"})
        results_df = results_df.merge(
            sub_audit, on=["dataset", "target", "split", "seed"], how="left"
        )
        results_df["collapsed"] = results_df["collapsed"].fillna(False)
        n_dropped = results_df["collapsed"].sum()
        if n_dropped:
            print(f"  Excluding {n_dropped} collapsed run(s) from aggregation "
                  f"(per collapse_audit.csv)")
        results_df = results_df[~results_df["collapsed"]].drop(columns="collapsed")
    else:
        print(f"  collapse_audit.csv not found at {audit_path} -- "
              f"run Collapse_Audit.ipynb after this notebook completes, "
              f"then re-run this cell.")

    raw_path = RESULTS_DIR / "all_seeds_raw.csv"
    results_df.to_csv(raw_path, index=False)
    print(f"  Raw results saved to {raw_path}")

    binary_df = results_df[results_df["target"] == "binary"]

    print("\n" + "=" * 70)
    print("  DENSE KAN — Binary Test AUC (mean ± std across seeds)")
    print("=" * 70 + "\n")

    if "test_auc" in binary_df.columns and len(binary_df) > 0:
        agg = binary_df.groupby(["dataset", "split"])["test_auc"].agg(
            ["mean", "std"]
        ).reset_index()
        agg["display"] = agg.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        pivot = agg.pivot(index="dataset", columns="split", values="display")
        grand = binary_df.groupby("dataset")["test_auc"].agg(["mean", "std"])
        pivot["Mean ± Std"] = grand.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        print(pivot.to_string())

    cont_df = results_df[results_df["target"] == "continuous"]

    print("\n" + "=" * 70)
    print("  DENSE KAN — Continuous Test R² (mean ± std across seeds)")
    print("=" * 70 + "\n")

    if "test_r2" in cont_df.columns and len(cont_df) > 0:
        agg = cont_df.groupby(["dataset", "split"])["test_r2"].agg(
            ["mean", "std"]
        ).reset_index()
        agg["display"] = agg.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        pivot = agg.pivot(index="dataset", columns="split", values="display")
        grand = cont_df.groupby("dataset")["test_r2"].agg(["mean", "std"])
        pivot["Mean ± Std"] = grand.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        print(pivot.to_string())

    print("\n" + "=" * 70)
    print("  DENSE KAN — Continuous Derived AUC (mean ± std across seeds)")
    print("=" * 70 + "\n")

    if "test_derived_auc" in cont_df.columns and len(cont_df) > 0:
        agg = cont_df.groupby(["dataset", "split"])["test_derived_auc"].agg(
            ["mean", "std"]
        ).reset_index()
        agg["display"] = agg.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        pivot = agg.pivot(index="dataset", columns="split", values="display")
        grand = cont_df.groupby("dataset")["test_derived_auc"].agg(["mean", "std"])
        pivot["Mean ± Std"] = grand.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        print(pivot.to_string())

    print("\n" + "=" * 70)
    print("  DENSE KAN — Continuous Test MSE (mean ± std across seeds)")
    print("=" * 70 + "\n")

    if "test_mse" in cont_df.columns and len(cont_df) > 0:
        agg = cont_df.groupby(["dataset", "split"])["test_mse"].agg(
            ["mean", "std"]
        ).reset_index()
        agg["display"] = agg.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        pivot = agg.pivot(index="dataset", columns="split", values="display")
        grand = cont_df.groupby("dataset")["test_mse"].agg(["mean", "std"])
        pivot["Mean ± Std"] = grand.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        print(pivot.to_string())
        print(f"\n  NOTE: MSE is in minret_5d_pct units (percentage points "
              f"squared), NOT z-score units.")

    print("\n" + "=" * 70)
    print("  CONTINUOUS: Prediction std (informational)")
    print("=" * 70 + "\n")

    if "test_pred_std" in cont_df.columns:
        for _, row in cont_df.iterrows():
            print(f"  seed={row['seed']}  {row['dataset']:<20} "
                  f"{row['split']:<10}  "
                  f"pred_std={row.get('test_pred_std', 0):.4f}")

    print("\n" + "=" * 70)
    print("  SEED STABILITY: Best hyperparameters across seeds")
    print("=" * 70 + "\n")

    print(f"  {'Seed':>5} {'Dataset':<20} {'Target':<12} {'Split':<10} "
          f"{'lr':>10} {'wd':>10} {'bs':>5} {'reg_w':>10}")
    print("  " + "-" * 85)
    for (seed, ds, tt, split), params in sorted(best_params_store.items()):
        print(f"  {seed:>5} {ds:<20} {tt:<12} {split:<10} "
              f"{params['lr']:>10.6f} {params['weight_decay']:>10.6f} "
              f"{params['batch_size']:>5} {params['reg_weight']:>10.2e}")

    print("\n" + "=" * 70)
    print("  SEED VARIANCE SUMMARY")
    print("=" * 70 + "\n")

    for tt in TARGET_TYPES:
        metric_col   = "test_auc" if tt == "binary" else "test_derived_auc"
        metric_label = "AUC"      if tt == "binary" else "Derived AUC"
        subset = results_df[results_df["target"] == tt]

        if metric_col not in subset.columns or len(subset) == 0:
            continue

        print(f"  {tt.upper()} ({metric_label}):")
        per_config = subset.groupby(["dataset", "split"])[metric_col].agg(
            ["mean", "std"]
        )
        max_std  = per_config["std"].max()
        mean_std = per_config["std"].mean()
        print(f"    Mean seed std across configs: {mean_std:.4f}")
        print(f"    Max seed std across configs:  {max_std:.4f}")

        if max_std < 0.01:
            print(f"    → Very stable: seed choice barely matters")
        elif max_std < 0.03:
            print(f"    → Moderately stable: some sensitivity to initialisation")
        else:
            print(f"    → High variance: results depend substantially on seed")
        print()

    print("\n" + "=" * 70)
    print("  TIMING")
    print("=" * 70 + "\n")

    for _, row in results_df.iterrows():
        print(f"  seed={row['seed']}  {row['dataset']:<20} "
              f"{row['split']:<10} {row['target']:<12} "
              f"best_epoch={row['best_epoch']:>3}  {row['time_s']:>6.1f}s")

    summary_rows = []
    for tt in TARGET_TYPES:
        subset = results_df[results_df["target"] == tt]
        if len(subset) == 0:
            continue

        metric_cols = [c for c in subset.columns
                       if c.startswith("test_") and
                       subset[c].dtype in [np.float64, np.float32, float]]

        agg = subset.groupby(["dataset", "split"])[metric_cols].agg(
            ["mean", "std"]
        ).reset_index()

        agg.columns = [
            f"{c[0]}_{c[1]}" if c[1] else c[0]
            for c in agg.columns
        ]

        agg["target"] = tt
        summary_rows.append(agg)

    if summary_rows:
        summary_df  = pd.concat(summary_rows, ignore_index=True)
        summary_path = RESULTS_DIR / "cross_seed_summary.csv"
        summary_df.to_csv(summary_path, index=False)
        print(f"\n  Cross-seed summary saved to {summary_path}")

else:
    print("\n  No results to display — all experiments failed.")


# %% [markdown]
# ## File Inventory

# %%
print("\n" + "=" * 70)
print("  SAVED FILES (on Google Drive)")
print("=" * 70)

for seed in SEEDS:
    seed_dir = RESULTS_DIR / f"seed_{seed}"
    print(f"\n  ── seed_{seed}/ ──")
    for subdir in ["predictions", "metrics", "checkpoints", "optuna"]:
        d = seed_dir / subdir
        if d.exists():
            files = list(d.glob("dense_kan_*"))
            print(f"    {subdir}/: {len(files)} files")
        else:
            print(f"    {subdir}/: (not yet created)")

for fname in ["all_seeds_raw.csv", "cross_seed_summary.csv"]:
    fpath = RESULTS_DIR / fname
    if fpath.exists():
        print(f"\n  {fname}: ✓")
    else:
        print(f"\n  {fname}: (not yet created)")


# %% [markdown]
# ## Backtests (Seed-Averaged Signal)
#
# NOTE: evaluation.py's fixed continuous threshold grids still assume a
# roughly [-5, +2] z-score range from the old target -- grid-edge
# warnings here reflect that KNOWN, DEFERRED issue, not a new bug.

# %%
def load_averaged_predictions(model_name, split_name, target_type, part,
                               seeds, results_dir):
    signals = []
    returns = None

    for seed in seeds:
        seed_dir = results_dir / f"seed_{seed}"
        loaded = load_predictions(
            model_name=model_name,
            split_name=split_name,
            target_type=target_type,
            part=part,
            results_dir=seed_dir,
        )
        preds = loaded["predictions"]

        if returns is None:
            returns = preds["daily_return"].values

        if target_type == "binary":
            signals.append(preds["y_prob"].values)
        else:
            signals.append(preds["y_pred"].values)

    avg_signal = np.mean(np.stack(signals, axis=0), axis=0)
    return returns, avg_signal


def _make_json_safe(obj):
    if isinstance(obj, dict): return {k: _make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, pd.DataFrame): return obj.reset_index().to_dict(orient="records")
    if isinstance(obj, (np.floating, np.integer)): return obj.item()
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj


# %%
print("\n" + "=" * 70)
print("  DENSE KAN — BACKTESTS (seed-averaged signal)")
print("  Signal = mean prediction across seeds 42, 123, 456")
print("=" * 70)

backtest_dir = RESULTS_DIR / "backtests"
backtest_dir.mkdir(parents=True, exist_ok=True)

backtest_rows    = []
backtest_records = {}

for dataset in DATASETS:
    for split_name in ALL_SPLITS:
        model_name = f"dense_kan_{dataset}"

        val_ret,  val_sig  = load_averaged_predictions(
            model_name, split_name, "binary", "val",  SEEDS, RESULTS_DIR)
        test_ret, test_sig = load_averaged_predictions(
            model_name, split_name, "binary", "test", SEEDS, RESULTS_DIR)

        bt_binary = run_full_backtest(
            val_returns=val_ret,   val_signal=val_sig,
            test_returns=test_ret, test_signal=test_sig,
            go_cash_when="above",
            model_name=f"{model_name} (binary)",
            split_name=split_name,
        )

        val_ret,  val_sig  = load_averaged_predictions(
            model_name, split_name, "continuous", "val",  SEEDS, RESULTS_DIR)
        test_ret, test_sig = load_averaged_predictions(
            model_name, split_name, "continuous", "test", SEEDS, RESULTS_DIR)

        bt_continuous = run_full_backtest(
            val_returns=val_ret,   val_signal=val_sig,
            test_returns=test_ret, test_signal=test_sig,
            go_cash_when="below",
            model_name=f"{model_name} (continuous)",
            split_name=split_name,
        )

        key = f"{dataset}/{split_name}"
        backtest_records[key] = {"binary": bt_binary, "continuous": bt_continuous}

        for target_type, bt in [("binary", bt_binary), ("continuous", bt_continuous)]:
            for strategy_name, strategy_key in [("simple", "simple"), ("risk_scaled", "risk_scaled")]:
                bt_result = bt[strategy_key]
                backtest_rows.append({
                    "dataset": dataset, "split": split_name,
                    "target_type": target_type, "strategy": strategy_name,
                    "sharpe": bt_result["sharpe"], "sortino": bt_result["sortino"],
                    "annual_return": bt_result["annual_return"],
                    "max_drawdown": bt_result["max_drawdown"],
                    "cumulative_return": bt_result["cumulative_return"],
                    "avg_exposure": bt_result["avg_exposure"],
                    "annual_turnover": bt_result["annual_turnover"],
                    "buy_hold_sharpe": bt_result["buy_hold_sharpe"],
                    "buy_hold_sortino": bt_result["buy_hold_sortino"],
                    "buy_hold_cumulative": bt_result["buy_hold_cumulative"],
                })

backtest_summary_df = pd.DataFrame(backtest_rows)
backtest_summary_path = backtest_dir / "backtest_summary.csv"
backtest_summary_df.to_csv(backtest_summary_path, index=False)
print(f"\n  Backtest summary saved to {backtest_summary_path}")

backtest_json_path = backtest_dir / "backtest_full_results.json"
with open(backtest_json_path, "w") as f:
    json.dump(_make_json_safe(backtest_records), f, indent=2, default=str)
print(f"  Full backtest results saved to {backtest_json_path}")


# %% [markdown]
# ## Disconnect Runtime

# %%
print("All experiments complete. Disconnecting runtime...")
from google.colab import runtime
runtime.unassign()

Mounted at /content/drive
PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
  ARCHITECTURE WIDTHS (derived from taxonomy, per dataset)
  agg_full_moments     n_subthemes= 331   n_themes= 13
  agg_means            n_subthemes= 128   n_themes= 13
Device: Tesla T4 (CUDA)
  DENSE KAN: 3 Seeds × 4 Splits × 2 Datasets × 2 Targets = 48 runs
  Seeds: [42, 123, 456]
  agg_full_moments     widths=[n_feat, 331, 13, 1]   params=10,767,015  (incl. BatchNorm gamma/beta)
  agg_means            widths=[n_feat, 128, 13, 1]   params=1,427,831  (incl. BatchNorm gamma/beta)
  Grid: G=14, K=3, range=[-5.5, 5.5] (FIXED, no grid adaptation)
  BatchNorm1d between every KAN layer
  Loss: binary=BCEWithLogitsLoss, continuous=HuberLoss(delta=per-split, see huber_delta.json)
  Early stop: binary=AUC, continuous=R²  (both maximize)
  Optuna: 40 trials per run, seeded TPE sampler
  Collapse handling: detect-and-flag post-hoc (Collapse_Audit.ipynb), no retry
  Results saving to: /content/drive/MyDrive/Thesis/Data/Resul

[I 2026-08-14 23:55:22,770] A new study created in RDB with name: dense_kan_agg_full_moments_binary_Split_A_seed42


  Running 40 Optuna trials (0 already complete)


  0%|          | 0/40 [00:00<?, ?it/s]

  Optuna best AUC: 0.8989
  Best params: {'lr': 0.007391351416351659, 'weight_decay': 1.0091396161231072e-05, 'batch_size': 128, 'reg_weight': 0.0023821688120315115}
  Epoch    1 | Train loss 0.8653 | Val loss 0.5181  AUC 0.7665 | LR 7.4e-03
    [activations @ epoch 1]  L1 pre-BN max|x|=  14.90 std= 0.805  post-BN max|x|= 5.00 std=0.644  |  L2 pre-BN max|x|=   3.47 std= 0.568  post-BN max|x|= 2.72 std=0.579
  Epoch   20 | Train loss 0.3501 | Val loss 0.6296  AUC 0.5605 | LR 1.8e-03
    [activations @ epoch 20]  L1 pre-BN max|x|=   7.62 std= 0.191  post-BN max|x|= 5.00 std=0.168  |  L2 pre-BN max|x|=   1.47 std= 0.382  post-BN max|x|= 2.48 std=0.644
  Early stop at epoch 21. Best val AUC: 0.7665 at epoch 1
  Training complete in 7.5s

  Results (seed=42):
    Train AUC: 0.8145
    Val AUC:   0.7665
    Test AUC:  0.7250
    Gap:       +0.0895

  ✓ Completed 1/48  (7min elapsed, ~330min remaining)

────────────────────────────────────────────────────────────
  seed=42 / agg_full_moments 

...

  Optuna best R²: 0.0879
  Best params: {'lr': 0.006386663203668006, 'weight_decay': 1.2842837521627336e-05, 'batch_size': 64, 'reg_weight': 0.0037312221661217368}
  Epoch    1 | Train Huber 0.6742 | Val Huber 0.4536  MSE 0.9344  R² 0.0247 | LR 6.4e-03
    [activations @ epoch 1]  L1 pre-BN max|x|=   3.52 std= 0.222  post-BN max|x|= 5.00 std=0.443  |  L2 pre-BN max|x|=   2.82 std= 0.191  post-BN max|x|= 3.06 std=0.257
  Epoch   20 | Train Huber 0.2452 | Val Huber 0.5912  MSE 1.2194  R² -0.2727 | LR 3.2e-03
    [activations @ epoch 20]  L1 pre-BN max|x|=   5.07 std= 0.164  post-BN max|x|= 5.00 std=0.151  |  L2 pre-BN max|x|=   2.22 std= 0.182  post-BN max|x|= 4.74 std=0.397
  Early stop at epoch 29. Best val R²: 0.0504 at epoch 9
  Training complete in 11.1s

  Results (seed=456, huber_delta=2.4735):
    Train R²:    0.6232  (MSE=0.5734)
    Val R²:      0.0504  (MSE=0.9098)
    Test R²:     0.1778  (MSE=2.3040)
    Derived AUC: 0.7395
    Pred std:    0.7316  (collapse check deferred t

  0%|          | 0/40 [00:00<?, ?it/s]

  Optuna best R²: 0.2985
  Best params: {'lr': 0.004559609128977492, 'weight_decay': 1.3287680604771734e-05, 'batch_size': 64, 'reg_weight': 0.00367905565323567}
  Epoch    1 | Train Huber 0.6416 | Val Huber 0.8004  MSE 1.8827  R² 0.3332 | LR 4.6e-03
    [activations @ epoch 1]  L1 pre-BN max|x|=   6.57 std= 0.292  post-BN max|x|= 5.00 std=0.504  |  L2 pre-BN max|x|=   3.92 std= 0.413  post-BN max|x|= 5.00 std=0.679
  Epoch   20 | Train Huber 0.2200 | Val Huber 1.3566  MSE 3.8585  R² -0.3665 | LR 1.1e-03
    [activations @ epoch 20]  L1 pre-BN max|x|=   2.81 std= 0.069  post-BN max|x|= 4.50 std=0.131  |  L2 pre-BN max|x|=   1.91 std= 0.154  post-BN max|x|= 4.64 std=0.389
  Early stop at epoch 21. Best val R²: 0.3332 at epoch 1
  Training complete in 9.1s

  Results (seed=456, huber_delta=2.4230):
    Train R²:    0.3282  (MSE=0.9627)
    Val R²:      0.3332  (MSE=1.8827)
    Test R²:     0.1902  (MSE=0.8210)
    Derived AUC: 0.7884
    Pred std:    0.4622  (collapse check deferred to C

  0%|          | 0/40 [00:00<?, ?it/s]

  Optuna best R²: 0.1880
  Best params: {'lr': 0.005620367507247949, 'weight_decay': 2.0290091368439596e-06, 'batch_size': 256, 'reg_weight': 0.0007277515876389584}
  Epoch    1 | Train Huber 0.9494 | Val Huber 1.0001  MSE 2.0282  R² -0.8049 | LR 5.6e-03
    [activations @ epoch 1]  L1 pre-BN max|x|=  14.24 std= 1.175  post-BN max|x|= 5.00 std=0.803  |  L2 pre-BN max|x|=   7.64 std= 0.904  post-BN max|x|= 5.00 std=0.761
  Epoch   20 | Train Huber 0.1428 | Val Huber 0.6836  MSE 1.3667  R² -0.2162 | LR 2.8e-03
    [activations @ epoch 20]  L1 pre-BN max|x|=   5.74 std= 0.249  post-BN max|x|= 5.00 std=0.337  |  L2 pre-BN max|x|=   4.20 std= 0.353  post-BN max|x|= 5.00 std=0.614
  Early stop at epoch 27. Best val R²: 0.0849 at epoch 7
  Training complete in 4.7s

  Results (seed=456, huber_delta=2.4998):
    Train R²:    0.4949  (MSE=0.8626)
    Val R²:      0.0849  (MSE=1.0284)
    Test R²:     -2.2391  (MSE=1.6052)
    Derived AUC: 0.7808
    Pred std:    0.7600  (collapse check deferred